# Comprensión y EDA — Riesgo de crédito

**Contexto de negocio:** base de creditos otorgados por una entidad financiera, con informacion
del credito, del cliente titular y del comportamiento de pago (`Pago_atiempo`). No se entrega
diccionario de datos, por lo que el entendimiento de cada variable se construye combinando
investigacion del negocio crediticio colombiano (central de riesgo tipo Datacredito) y validacion
empirica con los propios datos.

Este notebook documenta la limpieza, el entendimiento de negocio y el analisis exploratorio (EDA).
De aqui se derivan los criterios que despues se implementan como transformers reutilizables en
`ft_engineering.py`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_DATOS = "../../Base_de_datos.csv"

df = pd.read_csv(RUTA_DATOS, sep=';', encoding='utf-8-sig')
df.shape


In [ ]:
print(df.dtypes)
print()
print(df.isnull().sum())


## Hallazgo 1: la columna `puntaje`

1. Viene como texto, con coma decimal (formato es-CO), ej. `95,227787`.
2. El valor `95,227787` se repite en el 87% de las filas.
3. Existen valores negativos, algo atipico para un score de credito (Datacredito: 150-999).

**Hipotesis a validar:** `puntaje` no es un score de credito tradicional, sino una variable
contaminada con el resultado del pago (fuga de informacion / data leakage).


In [ ]:
df['puntaje_num'] = df['puntaje'].str.replace(',', '.', regex=False).astype(float)

print('--- Correlacion con puntaje_datacredito (score real) ---')
print(df[['puntaje_num', 'puntaje_datacredito']].corr())
print()

print('--- Correlacion con el target (Pago_atiempo) ---')
print(df['puntaje_num'].corr(df['Pago_atiempo']))
print()

mask_default = df['puntaje_num'].round(4) == 95.2278
print('--- Valor default vs mora ---')
print(pd.crosstab(mask_default, df['Pago_atiempo']))


### Conclusion

Correlacion `puntaje_num` vs `Pago_atiempo`: 0.92 (extrema). Correlacion con `puntaje_datacredito`
(score real): 0.09 (casi nula). El valor default nunca aparece en un credito en mora.

**Decision:** `puntaje` tiene fuga de informacion. Se documenta y se descarta como predictor
(implementado en `ColumnasIrrelevantes` dentro de `ft_engineering.py`).


## Hallazgo 2: valores numericos en `tendencia_ingresos`

58 filas (0.54%) tienen valores numericos en vez de Creciente/Decreciente/Estable.


In [ ]:
mask_anomalo = ~df['tendencia_ingresos'].isin(['Creciente', 'Decreciente', 'Estable']) & df['tendencia_ingresos'].notna()
print('Filas anomalas:', mask_anomalo.sum())

sub = df[mask_anomalo]
print('Nulos en promedio_ingresos_datacredito dentro de estas filas:', sub['promedio_ingresos_datacredito'].isna().sum(), 'de', len(sub))


### Conclusion

Se descarta corrimiento de columnas (todas las filas tienen 23 campos). Estas filas si tienen
dato de central de riesgo, por lo que deberian tener una tendencia valida -> son errores de
captura aislados.

**Decision:** convertir a nulo (implementado en `ColumnasNulos`).


## Hallazgo 3: nulos en variables de saldo

Patron jerarquico entre `saldo_mora`, `saldo_total`, `saldo_principal` y `saldo_mora_codeudor`.


In [ ]:
cols_saldo = ['saldo_mora', 'saldo_total', 'saldo_principal', 'saldo_mora_codeudor']
nulos = df[cols_saldo].isna()
print(nulos.value_counts())

grupo_todo_nulo = nulos['saldo_mora'] & nulos['saldo_total'] & nulos['saldo_principal'] & nulos['saldo_mora_codeudor']
print()
print('Filas con los 4 campos nulos:', grupo_todo_nulo.sum())
print('tipo_credito en ese grupo:', df.loc[grupo_todo_nulo, 'tipo_credito'].value_counts().to_dict())
print('tasa de mora en ese grupo (%):', round((1 - df.loc[grupo_todo_nulo, 'Pago_atiempo'].mean()) * 100, 2))


### Conclusion

- `saldo_mora_codeudor`: 99.97% de los no-nulos son 0 -> se interpreta como "sin codeudor" -> imputar con 0.
- `saldo_principal`: cuando `saldo_total` y `saldo_mora` existen, se deriva con la identidad contable
  `saldo_total ~= saldo_principal + saldo_mora`.
- 156 filas con los 4 campos nulos, concentradas en `tipo_credito=9` (97%) y con mas mora (7.05% vs 4.63%):
  no hay forma de reconstruirlos con evidencia.

**Decision:** implementado en la clase `Imputacion` de `ft_engineering.py`.


## Hallazgo 4: nulos en `promedio_ingresos_datacredito` / `tendencia_ingresos`

~2930 nulos "genuinos" (tras limpiar el Hallazgo 2), asociados a informalidad laboral.


In [ ]:
mask_prom = df['promedio_ingresos_datacredito'].isna()
mask_tend = df['tendencia_ingresos'].isna()
grupo = mask_prom & mask_tend

print('Coinciden en la misma fila:', grupo.sum())
print('puntaje_datacredito nulo en ese grupo:', df.loc[grupo, 'puntaje_datacredito'].isna().sum(), 'de', grupo.sum())
print()
print('tipo_laboral en el grupo con nulos:')
print(df.loc[grupo, 'tipo_laboral'].value_counts(normalize=True).round(3))
print()
print('tasa de mora grupo con nulos (%):', round((1 - df.loc[grupo, 'Pago_atiempo'].mean()) * 100, 2))
print('tasa de mora resto (%):', round((1 - df.loc[~grupo, 'Pago_atiempo'].mean()) * 100, 2))


### Conclusion

Estos clientes SI tienen score (no son nuevos), solo falta su info de ingresos. Se asocia
fuertemente a ser Independiente (56% vs 30%). En Colombia, el ingreso de un empleado formal lo
reporta el empleador; el de un independiente depende de fuentes mas limitadas -> no es error de
captura, es estructural.

**Decision:** no se imputa. Se crea la variable derivada `tiene_info_ingresos_buro` (en
`NuevasVariables`, `ft_engineering.py`).


## Hallazgo 5: `puntaje_datacredito` nulo (6 filas)


In [ ]:
sub = df[df['puntaje_datacredito'].isna()]
print(sub[['tipo_credito', 'edad_cliente', 'cant_creditosvigentes', 'huella_consulta']].to_string())


### Conclusion

`cant_creditosvigentes=0` y `huella_consulta=0` en las 6 filas: es su primer credito, logico que
no tengan score. **Decision:** se imputa con la mediana general como piso conservador (necesario
para que el modelo entrene sin NaN), documentado como limitacion (`Imputacion`).


## Hallazgo 6: codigos raros en `tipo_credito`


In [ ]:
resumen = df.groupby('tipo_credito')['Pago_atiempo'].agg(n='size', tasa_mora=lambda x: round((1 - x.mean()) * 100, 2))
print(resumen)


### Conclusion

Codigo 6 (n=21): 42.9% de mora, ~10x el promedio. Codigos 7 y 68 (n=2 y n=1): estadisticamente
inmanejables.

**Decision:** se crea `tipo_credito_agrupado` (4, 9, 10, 6 quedan visibles; 7+68 -> "Otros"),
implementado en `NuevasVariables`/`ToCategory`.


## Hallazgo 7: bloque de 150 filas con edad y salario distorsionados


In [ ]:
idx_bloque = df[df['edad_cliente'] > 90].index
print('Filas en el bloque:', len(idx_bloque))
print('Consecutivas?', (idx_bloque.max() - idx_bloque.min() + 1) == len(idx_bloque))

sub = df.loc[idx_bloque]
resto = df.loc[~df.index.isin(idx_bloque)]
for col in ['salario_cliente', 'total_otros_prestamos']:
    print(col, '- ratio mediana bloque/resto:', round(sub[col].median() / resto[col].median(), 1))


### Conclusion

Edad: patron matematico exacto (-100 anios, desviacion 0.2) -> se corrige. Salario y otros
prestamos: factores de escala distintos entre si (29.4x vs 16.1x) -> no se corrigen, se marcan
como no confiables.

**Decision:** implementado en la clase `Outliers` (`ft_engineering.py`), que corrige la edad y
crea la variable `lote_datos_sospechoso`.


## Hallazgo 8: `salario_cliente` invalido (35 filas)


In [ ]:
mask_cero = df['salario_cliente'] == 0
tmp = df[df['salario_cliente'] > 0]
ratio = tmp['cuota_pactada'] / tmp['salario_cliente']
mask_ratio = pd.Series(False, index=df.index)
mask_ratio.loc[ratio[ratio > 1].index] = True

print('Salario = 0:', mask_cero.sum())
print('Cuota > salario:', mask_ratio.sum())


### Conclusion

Salario en cero pero con credito real desembolsado = imposible en la practica (se exige verificar
capacidad de pago). Cuota mayor al salario, sin factor de escala consistente en 11 casos.

**Decision:** convertir a nulo unicamente en estas 35 filas puntuales (implementado en
`ColumnasNulos`).


## Analisis Exploratorio de Datos (EDA)

Con el entendimiento de negocio ya construido, se exploran las relaciones entre variables y el
target `Pago_atiempo`.


In [ ]:
print(df['Pago_atiempo'].value_counts())
print(df['Pago_atiempo'].value_counts(normalize=True).round(3))

plt.figure(figsize=(5,4))
df['Pago_atiempo'].value_counts().sort_index().plot(kind='bar', color=['#E63946', '#2E86AB'])
plt.title('Distribucion del target')
plt.xlabel('Pago_atiempo')
plt.ylabel('Numero de creditos')
plt.show()


**Insight:** 95.3% paga a tiempo, 4.7% cae en mora. Clases desbalanceadas -> hay que leer con
cuidado los segmentos con muestra chica (ej. el tipo_credito 6 del Hallazgo 6).


In [ ]:
bins_score = [0, 499, 649, 699, 749, 799, 999]
labels_score = ['Riesgo alto', 'Regular', 'Aceptable', 'Bueno', 'Muy bueno', 'Excelente']
df['rango_puntaje_datacredito'] = pd.cut(df['puntaje_datacredito'], bins=bins_score, labels=labels_score)

tabla = df.groupby('rango_puntaje_datacredito', observed=True)['Pago_atiempo'].agg(
    n='size', tasa_mora=lambda x: round((1 - x.mean()) * 100, 2)
)
print(tabla)

plt.figure(figsize=(7,4))
tabla['tasa_mora'].plot(kind='bar', color='#E63946')
plt.title('Tasa de mora por rango de puntaje Datacredito')
plt.ylabel('Tasa de mora (%)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


**Insight:** relacion monotonica y limpia: a mejor score, menor mora. Es el mejor predictor
individual encontrado en todo el analisis.


In [ ]:
idx_bloque = df[df['lote_datos_sospechoso'] == 1].index if 'lote_datos_sospechoso' in df.columns else df[df['edad_cliente'] > 90].index

df['edad_cliente_corr'] = df['edad_cliente']
df.loc[idx_bloque, 'edad_cliente_corr'] = df.loc[idx_bloque, 'edad_cliente'] - 100

bins_edad = [18, 25, 35, 45, 55, 100]
labels_edad = ['18-25', '26-35', '36-45', '46-55', '56+']
df['rango_edad'] = pd.cut(df['edad_cliente_corr'], bins=bins_edad, labels=labels_edad)

tabla_edad = df.groupby('rango_edad', observed=True)['Pago_atiempo'].agg(
    n='size', tasa_mora=lambda x: round((1 - x.mean()) * 100, 2)
)
print(tabla_edad)

plt.figure(figsize=(7,4))
tabla_edad['tasa_mora'].plot(kind='bar', color='#E63946')
plt.title('Tasa de mora por rango de edad (edad corregida)')
plt.ylabel('Tasa de mora (%)')
plt.tight_layout()
plt.show()


**Insight:** los clientes mas jovenes (18-25) tienen la mora mas alta, decreciendo con la edad
-> patron clasico de riesgo crediticio.


In [ ]:
tmp = df.copy()
tmp['huella_bin'] = pd.cut(tmp['huella_consulta'], bins=[-1, 0, 2, 4, 6, 100], labels=['0', '1-2', '3-4', '5-6', '7+'])
tabla_huella = tmp.groupby('huella_bin', observed=True)['Pago_atiempo'].agg(
    n='size', tasa_mora=lambda x: round((1 - x.mean()) * 100, 2)
)
print(tabla_huella)

plt.figure(figsize=(7,4))
tabla_huella['tasa_mora'].plot(kind='bar', color='#E63946')
plt.title('Tasa de mora segun huella de consulta reciente')
plt.ylabel('Tasa de mora (%)')
plt.tight_layout()
plt.show()


**Insight:** relacion monotonica creciente: mas consultas recientes = mas mora. Confirma la
hipotesis de negocio (buscar credito en muchas entidades a la vez = senal de estres financiero).

## Conclusiones generales del EDA

- `puntaje_datacredito` es el mejor predictor individual.
- `huella_consulta` y `edad_cliente` (corregida) tambien muestran patrones claros.
- El tipo de credito 6 es un segmento de alerta (42.9% de mora, muestra chica).
- La informalidad laboral se asocia a mayor riesgo, probablemente por menor trazabilidad.
- Estos hallazgos alimentan directamente las decisiones de `ft_engineering.py`.
